 # Notebook 4: LSTM Model for Human Activity Recognition

In [13]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader,TensorDataset
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report

In [14]:
base = r"C:\Users\M Mayavan\OneDrive"
base = base + r"\Desktop\HAR_Project\UCI HAR Dataset"
X_train = np.loadtxt(base + r"\train\X_train.txt")
X_test = np.loadtxt(base + r"\test\X_test.txt")
y_train = np.loadtxt(base + r"\train\y_train.txt") - 1
y_test = np.loadtxt(base + r"\test\y_test.txt") - 1

In [15]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
X_train = X_train.reshape(-1, 33, 17)
X_test = X_test.reshape(-1, 33, 17)
X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
y_test_t = torch.tensor(y_test, dtype=torch.long)

In [16]:
train_ds = TensorDataset(X_train_t, y_train_t)
test_ds = TensorDataset(X_test_t, y_test_t)
train_loader = DataLoader(train_ds,batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64)

In [18]:
class LSTM(nn.Module):
    def __init__(self):
        super(LSTM, self).__init__()
        self.lstm = nn.LSTM(17, 128,num_layers=2, batch_first=True,dropout=0.3)
        self.drop = nn.Dropout(0.5)
        self.fc = nn.Linear(128, 6)
    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        out = self.drop(out)
        out = self.fc(out)
        return out

In [19]:
model = LSTM()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)

In [20]:
for epoch in range(50):
    model.train()
    total_loss = 0 
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        output = model(X_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(epoch+1, total_loss)  

1 131.58351916074753
2 83.427581012249
3 62.3011334836483
4 43.852161556482315
5 30.786992087960243
6 26.640232518315315
7 21.609078131616116
8 18.6783982552588
9 16.915970351547003
10 15.201946184039116
11 13.263148503378034
12 11.508699338883162
13 12.43620759062469
14 10.64674404449761
15 10.008896545507014
16 8.672514455392957
17 9.77922469843179
18 8.126457587815821
19 7.2591775162145495
20 6.478782817721367
21 6.632399164605886
22 7.42323801945895
23 5.4356370549649
24 4.365659923991188
25 4.666133072692901
26 3.9220163878053427
27 4.029992673546076
28 2.8500664280727506
29 3.125647088396363
30 3.0399655394721776
31 2.743416902492754
32 4.622046709177084
33 3.4533134402008727
34 3.082149257999845
35 1.98342740139924
36 2.829849610454403
37 1.8291282875579782
38 2.766784857143648
39 2.537035984918475
40 1.7788638275233097
41 1.6886936263472307
42 1.952316512237303
43 2.508460728917271
44 1.054428089177236
45 1.1164688132994343
46 2.4986691523808986
47 1.5405205258866772
48 0.82114

In [21]:
model.eval()
all_preds = []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        output = model(X_batch)
        preds = torch.argmax(output,dim=1)
        all_preds.append(preds)
all_preds = torch.cat(all_preds)
print(classification_report(y_test_t,all_preds))        


              precision    recall  f1-score   support

           0       0.91      0.94      0.93       496
           1       0.89      0.92      0.91       471
           2       0.93      0.87      0.90       420
           3       0.96      0.86      0.91       491
           4       0.88      0.96      0.92       532
           5       1.00      1.00      1.00       537

    accuracy                           0.93      2947
   macro avg       0.93      0.93      0.93      2947
weighted avg       0.93      0.93      0.93      2947

